# Implementing volume selection.

## Load generated mesh

In [ ]:
import meshio
import numpy as np
from pyproj import Transformer
import matplotlib.pyplot as plt
import rasterio
import numpy as np

mesh = meshio.read(
    "/home/ebr/projects/release-volume-sampler/generated/messina_001/triangulation/triangulation.vtk",  # string, os.PathLike, or a buffer/open file
    # file_format="stl",  # optional if filename is a path; inferred from extension
    # see meshio-convert -h for all possible formats
)
# mesh.points, mesh.cells, mesh.cells_dict, ...

# mesh.vtk.read() is also possible

In [ ]:
mesh.points[:,0:-1]

In [ ]:
mesh.point_data
# Elevation: same as third component of point coordinates.
# is_boundary: Whether point is boundary or not.

In [ ]:
mesh.cells_dict # Triangles

In [ ]:
mesh.cell_data 
# is_interior: Only interior simplices cover the domain. Exterior simplices contains only exterior points.
# neighbours: Indices of neighbor simplices for each simplex. The kth neighbor is opposite to the kth vertex. For simplices at the boundary, -1 denotes no neighbor.

In [ ]:
# Compute p1, p2, p3 (3D)
UTM_epsg_code = 32633 #Messina strait
elevation = mesh.point_data["Elevation"]
transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{UTM_epsg_code}", always_xy=True)
easting, northing = transformer.transform(mesh.points[:,0], mesh.points[:,1])

In [ ]:
# Compute triangle normals
points = np.vstack([easting, northing, elevation]).T
triangles = mesh.cells_dict["triangle"]

p1 = points[triangles][:,0,:] # 1-th vertice.
p2 = points[triangles][:,1,:]
p3 = points[triangles][:,2,:]

cross_products = np.linalg.cross(p2-p1, p3-p1)

areas = 0.5 * np.linalg.norm(cross_products, axis=1) 
normals = cross_products/cross_products[:, 2].reshape(-1, 1) # scale so that n3 = 1. (Easier to calculate gradient)
sides = np.vstack([np.linalg.norm(s, axis=1) for s in [p3 - p2, p1 - p3, p2 - p1]]).T  # Compute side lengths # i-th side is opposite of i-th vertice.

In [ ]:
normals

In [ ]:
pi = points[triangles][:,:,:-1]

In [ ]:
pi[:,0,:]

In [ ]:
# Compute side normals (in plane).

points = np.vstack([easting, northing]).T
triangles = mesh.cells_dict["triangle"]

p1 = points[triangles][:,0,:]
p2 = points[triangles][:,1,:]
p3 = points[triangles][:,2,:]

# counterclockwise side vector ri opposite of i'th vertice
r3, r1, r2 = p2 - p1, p3 - p2, p1 - p3

# Normalize
r1 = np.divide(r1, np.sqrt(np.sum(r1**2, axis=1)).reshape((-1,1)))
r2 = np.divide(r2, np.sqrt(np.sum(r2**2, axis=1)).reshape((-1,1)))
r3 = np.divide(r3, np.sqrt(np.sum(r3**2, axis=1)).reshape((-1,1)))
 
R = np.array([[0, -1],[1, 0]])

# Side normals (si pointing towards i'th neighbour)
s1 = np.matmul(R, r1.T).T
s2 = np.matmul(R, r2.T).T
s3 = np.matmul(R, r3.T).T

In [ ]:
side_normals = np.stack([s1, s2, s3])

In [ ]:
grad = np.sum(normals[:, :-1] * side_normals, axis=2).T

In [ ]:
# Directional gradients across boundary
grad1 = np.sum(normals[:, :-1] * side_normals[0], axis=1)
grad2 = np.sum(normals[:, :-1] * side_normals[1], axis=1)
grad3 = np.sum(normals[:, :-1] * side_normals[2], axis=1)

In [ ]:
gradX = np.vstack([grad1, grad2, grad3]).T

In [ ]:
grad.shape

In [ ]:
gradX.shape

In [ ]:
side_normals[0]

In [ ]:
s1

In [ ]:
# Directional gradients across boundary
grad1 = np.sum(normal[:, :-1] * s1, axis=1)
grad2 = np.sum(normal[:, :-1] * s2, axis=1)
grad3 = np.sum(normal[:, :-1] * s3, axis=1)

In [ ]:
# Check range of values.
plt.hist(grad3, range=(-0.5,0.5), bins=30)

In [ ]:
# A triangle i is upstream of triangle j if grad_ji (gradient of triangle j in direction of i)
# is positive and grad_ij is negative.
grads = np.vstack([grad1, grad2, grad3]).T

In [ ]:
# neighbours: Indices of neighbor simplices for each simplex. The kth neighbor is opposite to the kth vertex. For simplices at the boundary, -1 denotes no neighbor.
neighbours = mesh.cell_data["neighbours"][0]

# Recursive selection of neighbouring cells.

1. Need to determine upstream neighbours (normal vector, exterior boudary normals. side lengths.)
1. Need to calculate FOS for each triangle (How to deal with uncertainty? - apply quantile...) Load raster and average?
1. Recursive generation of volumes.

In [ ]:
# Load FOS values.
fos_file = "/home/ebr/projects/release-volume-sampler/generated/messina_001/fos/quantiles/fos_quantiles_2.tif" # 0.5 quantile
tri_mask_path = "/home/ebr/projects/release-volume-sampler/generated/messina_001/triangulation/triangulation_raster.tif"

with rasterio.open(fos_file) as src:
    logfos = src.read(1)  # Read the triangle mask
    fos_profile = src.profile  # Copy metadata to use in output

with rasterio.open(tri_mask_path) as src:
    tri_mask = src.read(1)  # Read the triangle mask
    profile = src.profile  # Copy metadata to use in output

In [ ]:
n_triangles = triangles.shape[0]

# Set fos to high value where missing.
triangle_fos = np.array([np.nanmin(10**logfos[tri_mask == tri_index], initial=9999) for tri_index in range(n_triangles)])

In [ ]:
plt.hist(triangle_fos, range=(1, 20), bins=40)

In [ ]:
# Check that triangle is interior
is_interior = mesh.cell_data["is_interior"][0] == 1

In [ ]:
release_threshold = 1.4

In [ ]:
def will_be_released(triangle, release_volume):
    # Check conditions that the triangle will be relased.
    neighbours_are_released = [t in release_volume for t in neighbours[triangle]]
    
    # Fraction of perimter that is allready contained in relase volume.
    delta = sides[triangle][neighbours_are_released].sum()/sides[triangle].sum()
    
    return triangle_fos[triangle]*(1-delta) < release_threshold


def get_upstream_triangles(released_triangle):
    upstream_triangles = neighbours[released_triangle, grads[released_triangle,:] > 0]
    released_is_downstream = grads[upstream_triangles][neighbours[upstream_triangles] == released_triangle] < 0
    released_is_interior = is_interior[upstream_triangles]
    return(upstream_triangles[released_is_downstream & released_is_interior])


def get_release_volume(init_triangle):
    release_volume = []
    released_triangles = [init_triangle]
    
    while released_triangles:
        released_triangle = released_triangles.pop()
        release_volume.append(int(released_triangle))
        upstream_triangles = get_upstream_triangles(released_triangle)
        
        # Check which of the upstream triangles will be released
        upstream_is_released = [will_be_released(triangle, release_volume) for triangle in upstream_triangles]
        released_triangles.extend(upstream_triangles[upstream_is_released])
    return(release_volume)
    

In [ ]:
seed_triangles = np.arange(n_triangles)[triangle_fos < release_threshold]

In [ ]:
init_triangle = seed_triangles[0]
release_volumes = []

for seed_triangle in seed_triangles:
    release_volumes.append(get_release_volume(seed_triangle))

Get raster using mask.

In [ ]:

def write_volume_to_file(tri_mask_path, volume_path, triangle_indices):
    """
    Creates a binary mask for specified triangles and writes it to a new raster file.
    
    Parameters:
    - tri_mask_path (str): Path to the input raster file containing triangle indices.
    - volume_path (str): Path to the output volume raster file.
    - triangle_indices (list of int): List of triangle indices to include in the volume.
    """
    with rasterio.open(tri_mask_path) as src:
        tri_mask = src.read(1)  # Read the triangle mask
        profile = src.profile  # Copy metadata to use in output

    # Create binary volume mask: 1 if pixel belongs to specified triangles, else 0
    volume_mask = np.isin(tri_mask, triangle_indices).astype(np.uint8)

    # Update profile for single-band, unsigned 8-bit data
    profile.update(dtype=rasterio.uint8, count=1)

    with rasterio.open(volume_path, 'w', **profile) as dst:
        dst.write(volume_mask, 1)  # Write the volume mask to the output file

In [ ]:
seed_triangle_nr = 3
seed_triangle = seed_triangles[seed_triangle_nr]
nr_of_triangles=len(release_volumes[seed_triangle_nr])

print(seed_triangle, nr_of_triangles)
volume_mask_path = f"/home/ebr/projects/release-volume-sampler/generated/messina_001/triangulation/volumes/volume_{seed_triangle}.tif"

write_volume_to_file(tri_mask_path, volume_mask_path, release_volumes[seed_triangle_nr])

In [ ]:

        # Load triangulation mask.
        #logger.info(f"Load triangulation mask: {self.tri_mask_path}")
        #with rasterio.open(self.tri_mask_path) as src:
        #    self.tri_mask = src.read(1)
    
    def write_volume_to_file(self, tri_mask_path, volume_path, triangle_indices):
        # Write binary mask of release volume to file
        logger.info(f"Write volume to file: {volume_path}")
        with rasterio.open(tri_mask_path) as src:
            tri_mask = src.read(1)
            profile = src.profile
        
        volume_mask = np.isin(tri_mask, triangle_indices).astype(np.uint8)
        profile.update(dtype=rasterio.uint8, count=1)
        
        with rasterio.open(volume_path, 'w', **profile) as dst:
            dst.write(volume_mask, 1)

    
    @staticmethod
    def _read_tif(fname):
        "Read .tif data and profile using rasterio."
        #logger.info(f"Read file: {fname}")
        with rasterio.open(fname) as src:
            #data = np.ma.masked_equal(src.read(1), src.nodata)
            data = src.read(1)
            msk = np.where(src.read_masks(1) == src.nodata, False, True)
            profile = src.profile.copy()
        return data, msk, profile

In [ ]:
from itertools import product

def subsets(upstream):
    """
    Generates all combinations of subsets for the given list.
    Each subset is represented as a list of the same length as upstream,
    where 1 indicates inclusion and 0 indicates exclusion.
    """
    n = len(upstream)
    # Create all binary masks of length n
    for mask in product([False, True], repeat=n):
        yield np.array(mask)

In [ ]:
for i in subsets({1,2}):
    print(i)

In [ ]:
np.arange(5)

In [ ]:
np.array(list(set({1,4})))

In [ ]:
from itertools import product

upstream = np.array([3, 6, 19])
probs = np.array([0.3, 0.7, 0.1])

for sub in subsets(upstream):
    print(all(~sub))
    release = upstream[sub]
    print(len(release))
    prob_release = np.concat([(1-probs)[~sub], probs[sub]]).prod()

In [ ]:
len(release)

In [ ]:
for i,j in zip([1,3], [True, False]):
    print(i,j)

In [ ]:

    def _probability_of_release(self, upstream_triangles, released_volume):
        # Determine if a triangle will be released based on its FOS and its neighbors
        
        # Calculate reduction factor delta.
        # Perimeter of upstream_triangles (remove neighbouring sides)
        logger.info(f"_probability_of_release - upstream_triangles: {upstream_triangles}")
        perimeter_of_upstream = 0
        for triangle in upstream_triangles:
            neighbour_not_in_upstream_triangles = [neighbour not in upstream_triangles for neighbour in self.neighbours[triangle]]
            perimeter_of_upstream += self.sides[triangle][neighbour_not_in_upstream_triangles].sum()
        
        perimeter_of_upstream_released = 0
        for triangle in upstream_triangles:
            neighbours_in_release = [t in released_volume for t in self.neighbours[triangle]]
            perimeter_of_upstream_released += self.sides[triangle][neighbours_in_release].sum()
        
        delta = perimeter_of_upstream_released/perimeter_of_upstream
        
        # Fos should be a replaced by cumulative probability P(fos < threshold/(1-delta))
        # Calculate area weighted fos.
        area_weights = self.areas[upstream_triangles]/self.areas[upstream_triangles].sum()
        weighted_fos = np.average(self.triangle_fos[upstream_triangles], weights=area_weights)
        #return weighted_fos < self.fos_threshold/(1 - delta)
        return weighted_fos < self.fos_threshold/(1 - delta)

In [ ]:
import rasterio
import numpy as np
from shapely.geometry import Polygon, Point
import matplotlib.pyplot as plt

slumap_file = "/home/ebr/projects/release-volume-sampler/generated/slopeunits/slumap.tif"
tri_mask_file = "/home/ebr/projects/release-volume-sampler/generated/messina_001/triangulation/triangulation_raster.tif"

In [ ]:
np.arange(src.width)

In [ ]:
# Load the input raster with slope data
src = rasterio.open(slumap_file)

# Initialize the output raster array with nodata (-1)
triangle_mask = np.full((src.height, src.width), -1, dtype=np.int32)
triangles = mesh.cells_dict["triangle"] # Triangles
points = mesh.points[:,:-1] # lonlat coordinates.
row_indices, col_indices = np.meshgrid(np.arange(src.height), np.arange(src.width), indexing='ij')

lons, lats = rasterio.transform.xy(src.transform, row_indices, col_indices)


In [ ]:
lons, lats = np.array(lons), np.array(lats)

In [ ]:
lons.shape
triangle_mask.shape

In [ ]:
for i, triangle in enumerate(triangles):
    print(i, triangle)
    # Get the coordinates of the vertices of the triangle
    triangle_coords = points[triangle]
    polygon = Polygon(triangle_coords)
    left, lower, right, upper = polygon.bounds
    window_box_mask = (lons >= left) & (lons <= right) & (lats <= upper) & (lats >= lower)
    window_mask_triangle_flat = [polygon.contains(Point(lon, lat)) for lon, lat in zip(lons[window_box_mask].flatten(), lats[window_box_mask].flatten())]
    triangle_mask[window_box_mask] = np.where(window_mask_triangle_flat, i, triangle_mask[window_box_mask])

In [ ]:
np.array([polygon.contains(Point(lon, lat)) for lon, lat in zip(lons[t_mask].flatten(), lats[t_mask].flatten())])

In [ ]:
plt.imshow(triangle_mask, cmap="tab20b")
plt.colorbar()

In [ ]:
with rasterio.open(
    'triangulation_raster.tif',
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype=raster.dtype,
    crs=slope_crs,
    transform=slope_transform,
    nodata=-1
) as dst:
    dst.write(raster, 1)

In [ ]:
for lon in lons
        for col in range(min_col, max_col + 1):
            x, y = lons(col, row), lats(col, row)  # Convert pixel to lon/lat
            if polygon.contains(Point(x, y)):
                triangle_mask[row, col] = i + 1  # Assign a unique ID to the triangle

# Load cumulative probability.

We need the cummulative probability function for each triangle.

In [ ]:
import json, os

In [ ]:
def read_tif(fname):
    "Read .tif data and profile using rasterio."
    #logger.info(f"Read file: {fname}")
    with rasterio.open(fname) as src:
        #data = np.ma.masked_equal(src.read(1), src.nodata)
        data = src.read(1)
        msk = np.where(src.read_masks(1) == src.nodata, False, True)
        profile = src.profile.copy()
    return data, msk, profile

In [ ]:
cummulative_dir = "/home/ebr/projects/release-volume-sampler/generated/messina_001/fos/cummulative"
tri_mask_path = "/home/ebr/projects/release-volume-sampler/generated/messina_001/triangulation/triangulation_raster.tif"
n_triangles = 4000

# load triangulation mask

with rasterio.open(tri_mask_path) as src:
    tri_mask = src.read(1)

# load json file
with open(os.path.join(cummulative_dir, "content.json"),'r') as f:
    content = json.load(f)

# Initialize lists to store raster data and no-data values
rasters = []
nodata_vals = []

# Read all rasters and store the data
for e in content:
    raster_path = os.path.join(cummulative_dir, e["file"])
    raster_data, msk, profile = read_tif(raster_path)
    rasters.append(raster_data)

In [ ]:
triangle_cummulative_probs = np.empty((n_triangles, len(content)))
thresholds = []

for raster_index, e in enumerate(content):
    thresholds.append(e['threshold'])
    for tri_index in range(n_triangles):
        triangle_cummulative_probs[tri_index, raster_index] = np.nanmin(rasters[raster_index][tri_mask == tri_index], initial=999)
        
thresholds = np.array(thresholds)

In [ ]:
triangle_cummulative_probs[2000,:]

In [ ]:
thresholds

In [ ]:
# Evaluate.
triangle = 2000
threshold = 1.23

np.interp(threshold, xp = thresholds, fp = triangle_cummulative_probs[triangle,:], left=0., right=1.)

In [ ]:
plt.imshow(triangle_cummulative_probs,  aspect='auto', vmin=0, vmax=1)
plt.colorbar()

In [ ]:
tri_mask.max()

In [ ]:
# Load lookuptable
cumprob_logfos = np.load("/home/ebr/projects/release-volume-sampler/generated/messina_001/triangulation/cummulative_fos.npz")

In [ ]:
cumprob_logfos["thresholds"]

In [ ]:
cumprob_logfos["cummulative_probs"].shape